# IBM API sandbox

Scratch notebook for trying the IBM AI services the Crispr backend may use, before
any of it goes into the service itself.

**Where to run it:** the `jupyter-lab-ml` container on our LinuxONE Community Cloud
server (`http://<server-ip>:38888`). Upload it with
`scp -i key.pem notebooks/ibm_api_sandbox.ipynb linux1@<server-ip>:~/shared/`.

| Section | What it tests | Needs |
| --- | --- | --- |
| 1 | The machine: s390x, Python libraries, Telum AI accelerator (NNPA) | Nothing |
| 2 | **IBM Granite running locally** on LinuxONE via llama.cpp | The Granite model bundled in the image |
| 3 | **IBM watsonx.ai** hosted models over the REST API | A watsonx.ai API key and project ID (optional) |
| 4 | Outbound access to the genetics databases the backend will call | Internet access from the server |

Every section skips itself cleanly if its prerequisites are missing, so you can
**Run All** and read the summary at the end.

> Don't paste API keys into cells: they get saved in the notebook. Use environment
> variables or the `getpass` prompt in section 3.

In [ ]:
import os
import platform
import sys
import time

# Filled in by each section; printed at the end.
results = {}

## 1. Environment

LinuxONE is an **s390x** (big-endian) machine. Python wheels must be built for it,
which is why we use IBM's prebuilt image instead of `pip install`-ing everything.

In [ ]:
import glob
import importlib.metadata as metadata

print(f"Python           : {sys.version.split()[0]}")
print(f"Architecture     : {platform.machine()}  (byte order: {sys.byteorder})")
print(f"OS               : {platform.platform()}")

for package in ["llama_cpp_python", "requests", "numpy", "pandas", "torch", "scikit-learn"]:
    try:
        print(f"{package:<17}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package:<17}: not installed")

# The Telum chip's on-chip AI accelerator is exposed to llama.cpp through zDNN.
zdnn_libs = glob.glob(os.path.join(sys.prefix, "lib*", "python3*", "site-packages", "llama_cpp", "lib", "libggml-zdnn*.so"))
print(f"zDNN/NNPA        : {'available' if zdnn_libs else 'not found (CPU only)'}")

on_linuxone = platform.machine() == "s390x"
results["environment"] = "LinuxONE (s390x)" if on_linuxone else f"not LinuxONE ({platform.machine()})"

## 2. IBM Granite on LinuxONE (local, no API key)

The `jupyter-lab-ml` image ships an IBM Granite model in GGUF format and a
llama.cpp build that uses the NNPA accelerator. This runs entirely on our server,
so nothing leaves the machine, which fits the "AI on IBM Z" story for judging.

The model path comes from the image's own demo package (`loan_demo`). If that moves,
set `GRANITE_GGUF` to a `.gguf` file path. Note: on s390x the model file must be a
big-endian build; a model downloaded for a laptop will fail to load.

In [ ]:
def find_granite_model():
    if os.environ.get("GRANITE_GGUF"):
        return os.environ["GRANITE_GGUF"]
    try:
        from loan_demo.rag_utils import MODEL_PATH  # bundled with the jupyter-lab-ml image
        if os.path.exists(MODEL_PATH):
            return MODEL_PATH
    except ImportError:
        pass
    for root in ["/home/ibm-user", "/models", "/opt/models"]:
        matches = glob.glob(os.path.join(root, "**", "*.gguf"), recursive=True)
        if matches:
            return matches[0]
    return None


granite_path = find_granite_model()
print(f"Granite model: {granite_path or 'not found'}")
if granite_path:
    print(f"Size: {os.path.getsize(granite_path) / 1024**2:,.0f} MB")

In [ ]:
granite = None
if granite_path is None:
    results["granite_local"] = "skipped: no model file found"
else:
    try:
        from llama_cpp import Llama

        start = time.perf_counter()
        granite = Llama(model_path=granite_path, n_ctx=4096, verbose=False)
        print(f"Loaded in {time.perf_counter() - start:.1f}s")
    except Exception as error:
        results["granite_local"] = f"failed to load: {error}"
        print(results["granite_local"])

A prompt shaped like the backend's job: explain a known variant's effect in plain
language, without giving medical advice. Edit `VARIANT_QUESTION` to try others.

In [ ]:
SYSTEM_PROMPT = (
    "You are a genetics research assistant for a rare-disease research tool. "
    "Explain variant effects clearly for a non-specialist, say when evidence is "
    "uncertain, and never give medical advice."
)
VARIANT_QUESTION = (
    "In two or three sentences: what is the TP53 R175H variant, and what effect "
    "does it have on the p53 protein?"
)


def ask_granite(question, max_tokens=256):
    start = time.perf_counter()
    response = granite.create_chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        max_tokens=max_tokens,
        temperature=0.2,
    )
    elapsed = time.perf_counter() - start
    answer = response["choices"][0]["message"]["content"]
    tokens = response["usage"]["completion_tokens"]
    return answer, tokens, elapsed


if granite is not None:
    answer, tokens, elapsed = ask_granite(VARIANT_QUESTION)
    print(answer)
    print(f"\n{tokens} tokens in {elapsed:.1f}s ({tokens / elapsed:.1f} tokens/s)")
    results["granite_local"] = f"ok: {tokens / elapsed:.1f} tokens/s"

## 3. IBM watsonx.ai (hosted, optional)

Calls IBM's hosted models over plain REST with `requests`, so it needs no extra
packages on s390x. Skip this if the team has no watsonx.ai access.

You need:
- `WATSONX_APIKEY`: an IBM Cloud API key
- `WATSONX_PROJECT_ID`: the watsonx.ai project ID (project → Manage → General)
- `WATSONX_URL`: your region's endpoint, default `https://us-south.ml.cloud.ibm.com`

Set them as environment variables, or set `PROMPT_FOR_KEYS = True` to type them in.

In [ ]:
import getpass

import requests

PROMPT_FOR_KEYS = False
WATSONX_API_VERSION = "2024-05-01"

watsonx_url = os.environ.get("WATSONX_URL", "https://us-south.ml.cloud.ibm.com").rstrip("/")
watsonx_apikey = os.environ.get("WATSONX_APIKEY")
watsonx_project = os.environ.get("WATSONX_PROJECT_ID")
if PROMPT_FOR_KEYS and not (watsonx_apikey and watsonx_project):
    watsonx_apikey = getpass.getpass("IBM Cloud API key: ")
    watsonx_project = input("watsonx.ai project ID: ").strip()

watsonx_ready = bool(watsonx_apikey and watsonx_project)
if not watsonx_ready:
    results["watsonx"] = "skipped: no API key or project ID"
print(f"watsonx.ai endpoint: {watsonx_url}  configured: {watsonx_ready}")

In [ ]:
iam_token = None
if watsonx_ready:
    response = requests.post(
        "https://iam.cloud.ibm.com/identity/token",
        data={"grant_type": "urn:ibm:params:oauth:grant-type:apikey", "apikey": watsonx_apikey},
        timeout=30,
    )
    if response.ok:
        iam_token = response.json()["access_token"]
        print("IAM token: ok")
    else:
        results["watsonx"] = f"IAM auth failed: HTTP {response.status_code}"
        print(results["watsonx"], response.text[:300])

List the chat models available in this region, so we pick a real model ID instead
of guessing one.

In [ ]:
chat_models = []
if iam_token:
    response = requests.get(
        f"{watsonx_url}/ml/v1/foundation_model_specs",
        params={"version": WATSONX_API_VERSION, "filters": "function_text_chat", "limit": 200},
        timeout=30,
    )
    response.raise_for_status()
    chat_models = sorted(model["model_id"] for model in response.json()["resources"])
    print(f"{len(chat_models)} chat models:")
    for model_id in chat_models:
        print(" ", model_id)

In [ ]:
# Prefer an IBM Granite model; override by setting WATSONX_MODEL_ID.
watsonx_model = os.environ.get("WATSONX_MODEL_ID") or next(
    (model_id for model_id in chat_models if model_id.startswith("ibm/granite")),
    chat_models[0] if chat_models else None,
)

if iam_token and watsonx_model:
    start = time.perf_counter()
    response = requests.post(
        f"{watsonx_url}/ml/v1/text/chat",
        params={"version": WATSONX_API_VERSION},
        headers={"Authorization": f"Bearer {iam_token}"},
        json={
            "model_id": watsonx_model,
            "project_id": watsonx_project,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": VARIANT_QUESTION},
            ],
            "max_tokens": 256,
            "temperature": 0.2,
        },
        timeout=120,
    )
    elapsed = time.perf_counter() - start
    if response.ok:
        body = response.json()
        print(f"Model: {watsonx_model}\n")
        print(body["choices"][0]["message"]["content"])
        print(f"\n{body['usage']['completion_tokens']} tokens in {elapsed:.1f}s")
        results["watsonx"] = f"ok: {watsonx_model}"
    else:
        results["watsonx"] = f"chat failed: HTTP {response.status_code}"
        print(results["watsonx"], response.text[:500])

## 4. Genetics data sources reachable from LinuxONE

The backend will pull variant and structure data from public databases (see the
team's Hackathon doc). This only checks the server can reach them; it doesn't
download anything large.

In [ ]:
DATA_SOURCES = {
    "Ensembl REST": "https://rest.ensembl.org/info/ping?content-type=application/json",
    "NCBI E-utilities (ClinVar)": "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/einfo.fcgi?db=clinvar&retmode=json",
    "AlphaFold DB": "https://alphafold.ebi.ac.uk/api/prediction/P04637",
    "RCSB PDB": "https://data.rcsb.org/rest/v1/core/entry/2OCJ",
    "UniProt": "https://rest.uniprot.org/uniprotkb/P04637.json?fields=gene_names",
}

reachable = 0
for name, url in DATA_SOURCES.items():
    try:
        status = requests.get(url, timeout=15).status_code
    except requests.RequestException as error:
        status = type(error).__name__
    reachable += status == 200
    print(f"{name:<28} {status}")

results["data_sources"] = f"{reachable}/{len(DATA_SOURCES)} reachable"

## Summary

In [ ]:
for section, outcome in results.items():
    print(f"{section:<15} {outcome}")